In [0]:
##Need the dates when the satus gets changed from ordered to dispatched


print("-----GIVEN DATAFRAME-----\n\n\n")


data = [
    (1, "1-Jan", "Ordered"),
    (1, "2-Jan", "dispatched"),
    (1, "3-Jan", "dispatched"),
    (1, "4-Jan", "Shipped"),
    (1, "5-Jan", "Shipped"),
    (1, "6-Jan", "Delivered"),
    (2, "1-Jan", "Ordered"),
    (2, "2-Jan", "dispatched"),
    (2, "3-Jan", "shipped"),
]

columns = ["orderid", "statusdate", "status"]

df = spark.createDataFrame(data, columns)

df.show()


print("-----REQUIRED DATAFRAME-----\n\n\n")

data1 = [
    (1, "2-Jan", "dispatched"),
    (1, "3-Jan", "dispatched"),
    (2, "2-Jan", "dispatched"),
]

columns1 = ["orderid", "statusdate", "status"]

df1 = spark.createDataFrame(data1, columns1)

df1.show()




print("-----DSL SOLUTION-----\n\n\n")

##Using window function

from pyspark.sql.window import Window
from pyspark.sql.functions import lead, col

windowSpec = Window.partitionBy("orderid").orderBy("statusdate")

lead_df = (
    df.withColumn("status_change", lead(col("status"), 1).over(windowSpec))
      .filter(col("status") == "dispatched")
      .select("orderid", "statusdate", "status")
)

lead_df.show()

windowdf = df.withColumn("status_change", lead(col("status"), 1).over(windowSpec))


windowdf.show()

##Another solution using join


ordered_ids = df.filter(col("status") == "Ordered").select("orderid")
dispatched_ids = df.filter(col("status") == "dispatched")
result = df.filter(col("status") == "dispatched").join(ordered_ids, on="orderid", how="leftsemi")

result.show()


print("-----SQL SOLUTION-----\n\n\n")


df.createOrReplaceTempView("df_view")

spark.sql("""
    SELECT orderid, statusdate, status
    FROM (
        SELECT orderid, statusdate, status,
               Lead(status, 1) OVER (PARTITION BY orderid ORDER BY statusdate) AS status_change
        FROM df_view
    )
    WHERE status = 'dispatched'
""").show()


##Another solution using filter

spark.sql("""
          select * from df_view
          where status = 'dispatched' and orderid in (select orderid from df_view where status = 'Ordered')
          
          
          """).show()
